In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, avg, count, sum as _sum

spark = SparkSession.builder \
    .appName("Demo Viajes y Conductores") \
    .getOrCreate()

1. Leer archivos

In [2]:
viajes_df = spark.read.csv("viajes.csv", header=True, inferSchema=True)
conductores_df = spark.read.parquet("conductores.parquet")

print("Schema viajes:")
viajes_df.printSchema()

print("Schema conductores:")
conductores_df.printSchema()

print("Cantidad de viajes:", viajes_df.count())
print("Cantidad de conductores:", conductores_df.count())

Schema viajes:
root
 |-- id_viaje: integer (nullable = true)
 |-- id_conductor: integer (nullable = true)
 |-- ciudad: string (nullable = true)
 |-- distancia_km: double (nullable = true)
 |-- duracion_min: integer (nullable = true)
 |-- tarifa: integer (nullable = true)
 |-- fecha: date (nullable = true)

Schema conductores:
root
 |-- id_conductor: long (nullable = true)
 |-- nombre: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- tipo_vehiculo: string (nullable = true)
 |-- zona: string (nullable = true)

Cantidad de viajes: 1000000
Cantidad de conductores: 5000


2. Filtrado y ordenamiento encadenado

In [3]:
viajes_largos = viajes_df.filter(col("distancia_km") > 30).orderBy(col("tarifa").desc())

In [4]:
viajes_largos.show(10)

+--------+------------+-----------+------------+------------+------+----------+
|id_viaje|id_conductor|     ciudad|distancia_km|duracion_min|tarifa|     fecha|
+--------+------------+-----------+------------+------------+------+----------+
|  498674|        3675|Antofagasta|       45.99|         131| 41591|2025-12-30|
|  625980|         981|   Santiago|       45.99|         139| 41581|2026-03-19|
|  427217|        2218| Concepcion|       45.98|         134| 41578|2025-10-08|
|  917790|        2791|   Santiago|       45.98|         137| 41577|2025-09-25|
|  124848|        4849|  La Serena|       45.98|         134| 41576|2026-03-06|
|  635297|         298| Concepcion|       45.98|         141| 41564|2025-09-08|
|   76686|        1687| Valparaiso|       45.97|         137| 41562|2026-02-16|
|   15798|         799|  La Serena|       45.95|         141| 41553|2025-12-11|
|  326833|        1834|  La Serena|       45.96|         138| 41552|2025-10-17|
|  873020|        3021|   Santiago|     

In [5]:
print("Cantidad de viajes largos:", viajes_largos.count())

Cantidad de viajes largos: 355178


3. Agregaciones

In [6]:
resumen_ciudad = viajes_df.groupBy("ciudad") \
    .agg(
        count("*").alias("cantidad_viajes"),
        _sum("tarifa").alias("ingresos_totales"),
        avg("distancia_km").alias("distancia_promedio")
    ) \
    .orderBy(col("ingresos_totales").desc())

In [7]:
resumen_ciudad.show()

+-----------+---------------+----------------+------------------+
|     ciudad|cantidad_viajes|ingresos_totales|distancia_promedio|
+-----------+---------------+----------------+------------------+
|  La Serena|         200000|      4249412005|  23.5235734999999|
| Concepcion|         200000|      4247741087|23.513570800000014|
|Antofagasta|         200000|      4246060278| 23.50467965000033|
|   Santiago|         200000|      4240270548|23.470931699999777|
| Valparaiso|         200000|      4237842387|23.455789100000093|
+-----------+---------------+----------------+------------------+



agregación por tipo de recorrido

In [8]:
from pyspark.sql.functions import when, col, count, avg, sum as _sum # ya existe como función nativa por ello hace la diferencia.

viajes_segmentados = viajes_df.withColumn(
    "tipo_recorrido",
    when(col("distancia_km") < 10, "Corto")
    .when(col("distancia_km") < 30, "Medio")
    .otherwise("Largo")
)

resumen_recorrido = viajes_segmentados.groupBy("tipo_recorrido") \
    .agg(
        count("*").alias("cantidad_viajes"),
        _sum("tarifa").alias("tarifa_total"),
        avg("duracion_min").alias("duracion_promedio")
    ) \
    .orderBy(col("tarifa_total").desc())

In [9]:
resumen_recorrido.show()

+--------------+---------------+------------+------------------+
|tipo_recorrido|cantidad_viajes|tarifa_total| duracion_promedio|
+--------------+---------------+------------+------------------+
|         Largo|         355403| 11926030662|113.40225321677082|
|         Medio|         444865|  8112255962| 62.94692996751824|
|         Corto|         199732|  1183039681|22.385401437926824|
+--------------+---------------+------------+------------------+



4. Joins: inner y left

In [10]:
print("\n--- Inner join ---")
df_inner = viajes_df.join(conductores_df, on="id_conductor", how="inner")
df_inner.show(5)

print("\n--- Left join ---")
df_left = viajes_df.join(conductores_df, on="id_conductor", how="left")
df_left.show(5)


--- Inner join ---
+------------+--------+-----------+------------+------------+------+----------+-----------+------+-------------+--------+
|id_conductor|id_viaje|     ciudad|distancia_km|duracion_min|tarifa|     fecha|     nombre|rating|tipo_vehiculo|    zona|
+------------+--------+-----------+------------+------------+------+----------+-----------+------+-------------+--------+
|           2|  500001| Valparaiso|       18.04|          64| 17807|2025-05-12|Conductor_2|  3.53|    Hatchback|     Sur|
|           3|  500002| Concepcion|       16.35|          50| 14964|2025-05-11|Conductor_3|  4.69|       Pickup| Oriente|
|           4|  500003|  La Serena|        2.82|           8|  2932|2025-05-10|Conductor_4|  3.51|        Sedan|Poniente|
|           5|  500004|Antofagasta|        2.99|          22|  3569|2025-05-09|Conductor_5|  4.43|          SUV|  Centro|
|           6|  500005|   Santiago|       44.89|         137| 39251|2025-05-08|Conductor_6|  3.37|    Hatchback|   Norte|
+---

In [11]:
print("df_inner:", df_inner.count())
print("df_left:", df_left.count())

df_inner: 1000000
df_left: 1000000


5. selectExpr

In [12]:
df_expr = df_inner.selectExpr(
    "id_viaje",
    "nombre",
    "ciudad",
    "zona",
    "tipo_vehiculo",
    "tarifa",
    "distancia_km",
    "tarifa * 0.85 as tarifa_neta",
    "CASE WHEN distancia_km > 30 THEN 'Largo' ELSE 'Corto' END as tipo_recorrido",
    "CASE WHEN rating >= 4.5 THEN 'Top' ELSE 'Estandar' END as segmento_conductor"
)

df_expr.show(10)

+--------+------------+-----------+--------+-------------+------+------------+-----------+--------------+------------------+
|id_viaje|      nombre|     ciudad|    zona|tipo_vehiculo|tarifa|distancia_km|tarifa_neta|tipo_recorrido|segmento_conductor|
+--------+------------+-----------+--------+-------------+------+------------+-----------+--------------+------------------+
|  500001| Conductor_2| Valparaiso|     Sur|    Hatchback| 17807|       18.04|   15135.95|         Corto|          Estandar|
|  500002| Conductor_3| Concepcion| Oriente|       Pickup| 14964|       16.35|   12719.40|         Corto|               Top|
|  500003| Conductor_4|  La Serena|Poniente|        Sedan|  2932|        2.82|    2492.20|         Corto|          Estandar|
|  500004| Conductor_5|Antofagasta|  Centro|          SUV|  3569|        2.99|    3033.65|         Corto|          Estandar|
|  500005| Conductor_6|   Santiago|   Norte|    Hatchback| 39251|       44.89|   33363.35|         Largo|          Estandar|


6. Agregación más completa

In [17]:
reporte = df_inner.groupBy("ciudad", "tipo_vehiculo") \
    .agg(
        count("*").alias("cantidad_viajes"),
        _sum("tarifa").alias("tarifa_total"),
        avg("rating").alias("rating_promedio")
    ) \
    .orderBy(col("rating_promedio").desc())

reporte.show(20)

+-----------+-------------+---------------+------------+------------------+
|     ciudad|tipo_vehiculo|cantidad_viajes|tarifa_total|   rating_promedio|
+-----------+-------------+---------------+------------+------------------+
| Valparaiso|          SUV|          50000|  1058709491| 4.077679999999959|
|Antofagasta|        Sedan|          50000|  1063706997|4.0698800000000395|
|  La Serena|       Pickup|          50000|  1064387568| 4.052760000000019|
|  La Serena|          SUV|          50000|  1062707105| 4.052640000000102|
|Antofagasta|       Pickup|          50000|  1062370511| 4.048239999999927|
| Concepcion|    Hatchback|          50000|  1063348853| 4.033360000000035|
| Concepcion|       Pickup|          50000|  1062660079| 4.026440000000035|
| Valparaiso|       Pickup|          50000|  1061783915| 4.022599999999946|
| Valparaiso|    Hatchback|          50000|  1059034763| 4.022599999999865|
|   Santiago|        Sedan|          50000|  1058082151|4.0170799999998605|
|   Santiago

7. Visualizar plan de ejecución

In [18]:
from pyspark.sql.functions import col, count, avg, sum as _sum

demo_catalyst = viajes_df.select("ciudad", "tarifa", "distancia_km") \
    .filter(col("distancia_km") > 20) \
    .groupBy("ciudad") \
    .agg(
        count("*").alias("cantidad"),
        _sum("tarifa").alias("tarifa_total"),
        avg("distancia_km").alias("distancia_promedio")
    )

In [19]:
demo_catalyst.explain(True)

== Parsed Logical Plan ==
'Aggregate ['ciudad], ['ciudad, 'count(*) AS cantidad#598, 'sum('tarifa) AS tarifa_total#599, 'avg('distancia_km) AS distancia_promedio#600]
+- Filter (distancia_km#20 > cast(20 as double))
   +- Project [ciudad#19, tarifa#22, distancia_km#20]
      +- Relation [id_viaje#17,id_conductor#18,ciudad#19,distancia_km#20,duracion_min#21,tarifa#22,fecha#23] csv

== Analyzed Logical Plan ==
ciudad: string, cantidad: bigint, tarifa_total: bigint, distancia_promedio: double
Aggregate [ciudad#19], [ciudad#19, count(1) AS cantidad#598L, sum(tarifa#22) AS tarifa_total#599L, avg(distancia_km#20) AS distancia_promedio#600]
+- Filter (distancia_km#20 > cast(20 as double))
   +- Project [ciudad#19, tarifa#22, distancia_km#20]
      +- Relation [id_viaje#17,id_conductor#18,ciudad#19,distancia_km#20,duracion_min#21,tarifa#22,fecha#23] csv

== Optimized Logical Plan ==
Aggregate [ciudad#19], [ciudad#19, count(1) AS cantidad#598L, sum(tarifa#22) AS tarifa_total#599L, avg(distancia

In [20]:
spark.stop()